<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/mean_escape_time/ou_ball/ou_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ornstein–Uhlenbeck mean escape time: five-method comparison

This notebook solves the OU mean-exit-time equation in a disk and compares five neural formulations. It generates Monte Carlo labels, trains the selected methods, and evaluates each prediction against the radial reference solution.


## 1. Problem

In the disk $\Omega=B_R(0)\subset\mathbb R^2$, consider

$$dX_t=-\theta X_t\,dt+\sigma\,dB_t,$$

and the first exit time $T=\inf\{t\geq0:X_t\in\partial\Omega\}$. Its mean $\tau(x)=\mathbb E_x[T]$ solves

$$-\mathcal L\tau=1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

with $\mathcal L u=-\theta x\cdot\nabla u+\frac{\sigma^2}{2}\Delta u$.

The recorded experiment uses $R=2$, $\theta=3$, and **$\sigma^2=2$**. For $d$ dimensions, Kersting et al. give

$$
\tau(x)=\frac{1}{\lambda^{d/2}\sigma^2}
\int_{\lVert x\rVert}^{R}z^{1-d}e^{\lambda z^2}
\gamma\!\left(\frac d2,\lambda z^2\right)\,dz,
\qquad \lambda=\frac{\theta}{\sigma^2},
$$

where $\gamma(s,y)=\int_0^y t^{s-1}e^{-t}\,dt$. In two dimensions this becomes

$$
\tau(x)=\frac1\theta\int_{\lVert x\rVert}^{R}
\frac{e^{\lambda z^2}-1}{z}\,dz.
$$

At $r=0$, the integrand is defined by continuity, with value $0$. The code evaluates $e^{\lambda z^2}-1$ using `expm1`, which avoids cancellation near zero.


### Methods compared

All five methods use the same base network and differ in how they impose the boundary condition and define the losses. The standard method penalizes sampled boundary values; the variational method uses a weak residual and Nitsche boundary term; the other three impose zero values through different powers of the distance factor.

| Name in code | Trial function | Loss components |
|---|---|---|
| `standard` | $N_\theta$ | strong PDE, sampled boundary, Monte Carlo data |
| `adapted` | $dN_\theta$ | strong PDE, Monte Carlo data |
| `over` | $d^{3/2}N_\theta$ | strong PDE, Monte Carlo data |
| `under` | $d^{1/2}N_\theta$ | strong PDE, Monte Carlo data |
| `variational` | $N_\theta$ | weak residual, Nitsche boundary term, Monte Carlo data |

Here $d(x)=(R^2-\lVert x\rVert^2)/(2R)$ vanishes on the circle. The three exact-boundary methods use

$$u_\theta(x)=d(x)^\alpha N_\theta(x),\qquad
\alpha\in\{1,3/2,1/2\}.$$

The strong residual and Monte Carlo data losses are

$$J_{\rm pde}=\frac1{N_\Omega}\sum_i|\mathcal L u_\theta(x_i)+1|^2,
\qquad
J_{\rm data}=\frac1{N_d}\sum_j|u_\theta(z_j)-\widehat\tau_j|^2.$$

The standard PINN also uses $J_{\rm bc}=N_{\partial\Omega}^{-1}\sum_j|u_\theta(y_j)|^2$. Each active component $J_i$ is combined exactly as in the original notebooks,

$$J_{\rm train}=\sum_i e^{-2s_i}J_i+\sum_i s_i,$$

with trainable log scales $s_i$. The checkpoint with the smallest total training objective is retained.


In [ ]:
# Colab setup: this notebook works in Colab or a local Python environment.
import json
import math
import random
import secrets
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate
import torch
from torch import nn

# Runtime settings are separate from the experiment parameters below.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_NAME = "float64"  # Change to "float32" if needed.
if DTYPE_NAME not in ("float32", "float64"):
    raise ValueError("DTYPE_NAME must be 'float32' or 'float64'.")
torch.set_default_dtype(torch.float64 if DTYPE_NAME == "float64" else torch.float32)
print("Training device:", DEVICE)

## 2. Configure the experiment

Choose one run action below. `new` creates a unique folder, draws fresh seeds, prepares data, and trains. `resume` finds an unfinished saved run; `evaluate` loads completed models. Drive is mounted only when `SAVE_TO_DRIVE = True`; otherwise files go to `runs/ou` in the current runtime.

### Run and output

Set `RUN_MODE` to `"new"` for a fresh run, `"resume"` to continue an interrupted run, or `"evaluate"` to load saved models. A new run gets fresh recorded seeds and its own folder. In Colab, `SAVE_TO_DRIVE = False` keeps results in the runtime's `runs` folder; set it to `True` to mount Drive and save there.

In [ ]:
# Select a run action and where its files should go.
RUN_MODE = "new"                 # "new" trains a fresh run; "resume" or "evaluate" selects a saved run.
RUN_LABEL = "run"             # Name used to find runs in the chosen output folder.
SAVE_TO_DRIVE = False            # True mounts Google Drive when the run folder is prepared; False uses ./runs/ou.
METHODS = ["standard", "adapted", "over", "under", "variational"]
RUN_LBFGS = False                # Optional second optimization stage.

### OU process and exit domain

These parameters define the drift, diffusion, and exit boundary in the PDE above.

In [ ]:
R = 2.0                           # Radius of the absorbing disk.
theta = 3.0                        # Drift strength: dX = -theta X dt + sigma dW.
sigma_squared = 2.0                # Diffusion variance in the SDE.
sigma = math.sqrt(sigma_squared)

## 3. Model and PDE

The next cell uses ordinary PyTorch code. Change `BaseNetwork` directly to alter its layers, activations, features, forward pass, or initialization. It takes one coordinate per row and returns one scalar per row. The method class applies the boundary trial function and owns the loss weights.

In [ ]:
def smooth_distance(x):
    return (R**2 - x.square().sum(dim=1)) / (2 * R)


class BaseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        # PyTorch initializes each nn.Linear when it is created. To use another
        # initialization, write the nn.init_* calls here, for example:
        # for layer in self.layers:
        #     if isinstance(layer, nn.Linear):
        #         nn.init.xavier_normal_(layer.weight)
        #         nn.init.zeros_(layer.bias)


    def forward(self, points):
        return self.layers(points)


class OUModel(nn.Module):
    def __init__(self, method):
        super().__init__()
        self.method = method
        if method == "adapted":
            self.power = 1.0
        elif method == "over":
            self.power = 1.5
        elif method == "under":
            self.power = 0.5
        else:
            self.power = None
        self.network = BaseNetwork()
        if method == "standard":
            loss_names = ("pde", "data", "bc")
        elif method == "variational":
            loss_names = ("var", "data", "bc")
        else:
            loss_names = ("pde", "data")
        initial = {name: adaptive_log_scale_init for name in loss_names}
        if method == "variational":
            initial["bc"] = nitsche_bc_log_scale_init
        self.log_scales = nn.ParameterDict({
            name: nn.Parameter(torch.tensor(value))
            for name, value in initial.items()
        })

    def forward(self, x):
        raw = self.network(x)
        if raw.ndim == 1:
            raw = raw.unsqueeze(1)
        if raw.shape != (x.shape[0], 1):
            raise ValueError("BaseNetwork must return one scalar per point: [n] or [n, 1].")
        if self.power is None:
            return raw
        return raw * smooth_distance(x).clamp_min(0).pow(self.power).unsqueeze(1)


### PDE residual

For the OU generator shown above, the mean-exit-time residual is \(\mathcal L\tau_\theta+1\). Its coefficients are the same \(\theta\) and \(\sigma^2\) defined in the problem settings.

In [ ]:
def strong_residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    generator = -theta * (x * grad).sum(dim=1) + 0.5 * sigma_squared * (tau_xx + tau_yy)
    return generator + 1.0


## 4. Sampling and reference data

`training_collocation_points` and `training_boundary_points` are the total fixed sets used by the empirical PDE and boundary losses. The batch sizes below control how many saved points enter each Adam update. The full sets and separate evaluation sets are saved with the run.

In [ ]:
training_collocation_points = 512  # Total interior points used in the PDE loss each epoch.
training_boundary_points = 128     # Total boundary points used in the boundary loss each epoch.
evaluation_interior_points = 512   # Separate fixed points used only for progress reporting.
evaluation_boundary_points = 128

### Monte Carlo label settings

The code samples anchor points from the exit domain and estimates each mean exit time from `paths_per_anchor` (or `mc_paths`) independent Euler–Maruyama paths. `mc_dt` and `mc_max_time` control the time discretization and censoring cap. The anchors, estimates, and censored fractions are saved.

In [ ]:
trajectory_start = (0.0, 0.0)
trajectory_dt = 1e-3
trajectory_max_time = 20.0
trajectory_seed_offset = 1

### Analytic reference

The following cell evaluates the formula above. `expm1` preserves accuracy near the disk center.

In [ ]:
def true_tau_radius(radius):
    lam = theta / sigma_squared

    def integrand(z):
        return 0.0 if z == 0.0 else np.expm1(lam * z * z) / z

    value, _ = integrate.quad(integrand, float(radius), R, epsabs=1e-10, epsrel=1e-10)
    return value / theta


def true_tau(points):
    radii = np.linalg.norm(np.asarray(points), axis=1)
    return np.array([true_tau_radius(r) for r in radii])


grid_n = reference_grid_n
gx, gy = np.meshgrid(np.linspace(-R, R, grid_n), np.linspace(-R, R, grid_n))
disk_mask = gx**2 + gy**2 <= R**2
grid_points = np.column_stack((gx[disk_mask], gy[disk_mask]))
reference_grid = np.zeros_like(gx)
reference_grid[disk_mask] = true_tau(grid_points)

plt.figure(figsize=(5, 4))
plt.contourf(gx, gy, reference_grid, levels=50)
plt.colorbar(label=r"$\tau(x)$")
plt.gca().set_aspect("equal")
plt.title("OU reference mean escape time")
plt.show()


### Generate Monte Carlo labels

Interior anchors are sampled uniformly by area in the disk. From each anchor, independent paths follow the Euler–Maruyama update shown above until they leave the disk or reach `mc_max_time`. The label is their mean recorded exit time; capped paths contribute the time cap, and their fractions are saved with the labels.

In [ ]:
data_anchors = 24                   # Starting points where exit-time labels are estimated.
paths_per_anchor = 24               # Independent paths from each label point.
mc_dt = 5e-3                        # Euler-Maruyama time step.
mc_max_time = 2.0                   # Paths not exited by this time are censored.

## 5. Training objectives and controls

The optimizer is constructed explicitly in the training code. These values set its learning rate, Adam moments, weight decay, number of epochs, learning-rate decay, printed progress, and checkpoint frequency. Edit the optimizer or scheduler code directly to use a different algorithm.

In [ ]:
learning_rate = 5e-4
adam_beta_1 = 0.9
adam_beta_2 = 0.999
adam_epsilon = 1e-8
weight_decay = 0.0
adam_epochs = 40
final_learning_rate_factor = 1e-3
report_every = 10
checkpoint_every = 10

### Optional L-BFGS and Nitsche controls

L-BFGS is a second optimization stage and is off unless `RUN_LBFGS` is enabled. The variational/Nitsche controls apply only to that method. Change those values here and the corresponding loss code below.

In [ ]:
lbfgs_interior_points = 512
lbfgs_boundary_points = 512
lbfgs_iterations = 20
lbfgs_lr = 1.0
lbfgs_history_size = 1000
lbfgs_tolerance = 1e-9
n_test_functions = 50               # Used only by the variational/Nitsche method.
test_length = R / 5.0               # Width of its Gaussian test functions.
adaptive_log_scale_init = -0.5      # Initial adaptive loss weight; edit only for that method.
nitsche_bc_log_scale_init = 3.0     # Initial Nitsche boundary-loss weight.

### Evaluation settings

The grid size controls numerical evaluation and figure resolution. Predictions, reference values, signed errors, metrics, and figure files are saved under the run folder.

In [ ]:
reference_grid_n = 200
plot_dpi = 180

### Optional trajectory settings

This short simulation is for an optional illustration. It does not create training points or Monte Carlo labels.

In [ ]:
def simulate_ou_trajectory(
    start=(0.0, 0.0), *, theta_value=theta, sigma_squared_value=sigma_squared,
    radius_value=R, dt=1e-3, max_time=20.0, seed=SEED + 1,
):
    rng = np.random.default_rng(seed)
    point = np.asarray(start, dtype=float).copy()
    path = [point.copy()]
    exited = np.linalg.norm(point) >= radius_value
    for _ in range(int(max_time / dt)):
        if exited:
            break
        point = point - theta_value * point * dt + math.sqrt(sigma_squared_value * dt) * rng.standard_normal(2)
        path.append(point.copy())
        exited = np.linalg.norm(point) >= radius_value
    return np.asarray(path), (len(path) - 1) * dt, exited


trajectory, trajectory_time, trajectory_exited = simulate_ou_trajectory(
    start=trajectory_start, dt=trajectory_dt, max_time=trajectory_max_time,
    seed=SEED + trajectory_seed_offset,
)
angle = np.linspace(0.0, 2.0 * np.pi, 300)
fig, axis = plt.subplots(figsize=(5, 5))
axis.plot(trajectory[:, 0], trajectory[:, 1], lw=1.0)
axis.plot(R * np.cos(angle), R * np.sin(angle), "k--", label="exit boundary")
axis.scatter(*trajectory[0], label="start")
axis.scatter(*trajectory[-1], marker="x", label="exit" if trajectory_exited else "time cap")
axis.set(aspect="equal", title=f"OU path: {'exit' if trajectory_exited else 'censored'} at t={trajectory_time:.3g}")
axis.legend()
plt.show()


### Variational/Nitsche loss

Write $A=\sigma^2/2$. Multiplying $-\mathcal Lu=1$ by a test function $\phi$ and integrating the diffusion term by parts gives

$$
A\int_\Omega\nabla u\cdot\nabla\phi\,dx
+\theta\int_\Omega(x\cdot\nabla u)\phi\,dx
-A\int_{\partial\Omega}(\nabla u\cdot n)\phi\,ds
=\int_\Omega\phi\,dx.
$$

The symmetric Nitsche term is $-A\int_{\partial\Omega}u\,\nabla\phi\cdot n\,ds$; the raw penalty $\int_{\partial\Omega}u\phi\,ds$ is the separately weighted boundary component. The notebook uses 50 Gaussian tests

$$\phi_k(x)=\exp\!\left(-\frac{|x-c_k|^2}{2s^2}\right),\qquad s=R/5,$$

and estimates the integrals by sample averages.


In [ ]:
def sample_disk(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    radius = R * torch.sqrt(torch.rand(count, device=device))
    return torch.stack((radius * torch.cos(angle), radius * torch.sin(angle)), dim=1)


def sample_boundary(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    return torch.stack((R * torch.cos(angle), R * torch.sin(angle)), dim=1)


@torch.no_grad()
def monte_carlo_exit_time(x0, n_paths, dt, max_time):
    x0 = x0.to(DEVICE)
    batch = x0.shape[0]
    positions = x0[:, None, :].expand(batch, n_paths, 2).clone()
    elapsed = torch.zeros(batch, n_paths, device=DEVICE)
    alive = torch.ones(batch, n_paths, dtype=torch.bool, device=DEVICE)
    max_steps = int(max_time / dt)
    interval = max(1, max_steps // 100)
    noise_scale = sigma * math.sqrt(dt)
    started = time.time()

    for step in range(max_steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current - theta * current * dt + noise_scale * torch.randn_like(current)
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        exited = positions[alive].square().sum(dim=1) >= R**2
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == max_steps:
            print(
                f"MC step {step + 1:6d}/{max_steps} | "
                f"alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s",
                flush=True,
            )

    if alive.any():
        print(f"Censored at {max_time:g}: {int(alive.sum())}/{alive.numel()} paths")
    return elapsed.mean(dim=1), alive.float().mean(dim=1)


## 6. Prepare and train

A new run gets a unique folder and fresh training and Monte Carlo seeds. For resume or evaluation, the notebook finds saved runs automatically; if there is more than one, it lists them and asks for a run number. The selected path is stored in `run_dir` and used automatically by later cells.

In [ ]:
if RUN_MODE not in {"new", "resume", "evaluate"}:
    raise ValueError("RUN_MODE must be 'new', 'resume', or 'evaluate'.")

if SAVE_TO_DRIVE:
    if drive is None:
        raise RuntimeError("Google Drive saving is available in Colab; set SAVE_TO_DRIVE = False for local runs.")
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)
    output_base = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/ou")
else:
    output_base = Path("runs/ou")
output_base.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "new":
    # Each new run gets a fresh pair of seeds and a unique folder.
    SEED = secrets.randbelow(2**32)
    MC_SEED = secrets.randbelow(2**32)
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    output_root = output_base / f"ou_{RUN_LABEL}_{run_stamp}_seed{SEED}"
    output_root.mkdir(parents=False, exist_ok=False)
    saved_run = None
else:
    candidates = sorted(
        (path for path in output_base.glob(f"ou_{RUN_LABEL}_*") if path.is_dir() and (path / "run_manifest.json").is_file()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No saved OU run for {RUN_LABEL!r} was found in {output_base}.")
    if len(candidates) == 1:
        output_root = candidates[0]
    else:
        print("Choose the saved OU run:")
        for number, path in enumerate(candidates, start=1):
            print(f"  {number}. {path.name}")
        choice = int(input(f"Run number [1-{len(candidates)}]: "))
        if not 1 <= choice <= len(candidates):
            raise ValueError("Choose one of the listed run numbers.")
        output_root = candidates[choice - 1]
    run_config_path = output_root / "run_manifest.json"
    saved_run = json.loads(run_config_path.read_text(encoding="utf-8"))
    SEED = int(saved_run["seed"])
    MC_SEED = int(saved_run.get("mc_seed", saved_run.get("monte_carlo_seed")))

RESUME = RUN_MODE == "resume"
LOAD_SAVED_MODELS = RUN_MODE == "evaluate"
print(f"Mode: {RUN_MODE} | output: {output_root} | training seed: {SEED} | Monte Carlo seed: {MC_SEED}")


### Run record

The manifest records the values used for this run, its status, and the files it has saved. The next cells write and update it; edit the experiment values in their own sections above.

In [ ]:
test_centers = sample_disk(n_test_functions)


def variational_residual(model, interior, boundary):
    interior = interior.detach().clone().requires_grad_(True)
    boundary = boundary.detach().clone().requires_grad_(True)
    u_int = model(interior).squeeze(1)
    grad_u = torch.autograd.grad(u_int.sum(), interior, create_graph=True)[0]

    delta = interior[:, None, :] - test_centers[None, :, :]
    phi = torch.exp(-delta.square().sum(dim=2) / (2 * test_length**2))
    grad_phi = -delta * phi[..., None] / test_length**2
    area = math.pi * R**2
    diffusion = 0.5 * sigma_squared * (grad_u[:, None, :] * grad_phi).sum(dim=2)
    drift = theta * (interior * grad_u).sum(dim=1, keepdim=True) * phi
    weak = (diffusion + drift).mean(dim=0) * area - phi.mean(dim=0) * area

    delta_b = boundary[:, None, :] - test_centers[None, :, :]
    phi_b = torch.exp(-delta_b.square().sum(dim=2) / (2 * test_length**2))
    grad_phi_b = -delta_b * phi_b[..., None] / test_length**2
    normal = boundary / R
    u_b = model(boundary).squeeze(1)
    grad_u_b = torch.autograd.grad(u_b.sum(), boundary, create_graph=True)[0]
    arclength = 2 * math.pi * R
    flux_u = (grad_u_b * normal).sum(dim=1)
    flux_phi = (grad_phi_b * normal[:, None, :]).sum(dim=2)
    weak = weak - 0.5 * sigma_squared * (flux_u[:, None] * phi_b).mean(dim=0) * arclength
    symmetry = -0.5 * sigma_squared * (u_b[:, None] * flux_phi).mean(dim=0) * arclength
    raw_boundary = (u_b[:, None] * phi_b).mean(dim=0) * arclength
    return weak + symmetry, raw_boundary


def adaptive_objective(model, components):
    weighted_components = [
        torch.exp(-2 * model.log_scales[name]) * value + model.log_scales[name]
        for name, value in components.items()
    ]
    return sum(weighted_components)


def component_losses(model, method, interior, boundary, data_x, data_y):
    data_loss = (model(data_x).squeeze(1) - data_y).square().mean()
    if method == "variational":
        weak, raw_boundary = variational_residual(model, interior, boundary)
        return {"var": weak.square().mean(), "data": data_loss, "bc": raw_boundary.square().mean()}
    losses = {"pde": strong_residual(model, interior).square().mean(), "data": data_loss}
    if method == "standard":
        losses["bc"] = model(boundary).square().mean()
    return losses


### Prepare saved training data

A new run saves its Monte Carlo anchors, labels, censoring fractions, and fixed evaluation points in its own folder. Resume and evaluation load those arrays from the selected folder and reject mismatched settings.

In [ ]:
data_path = output_root / f"ou_{RUN_LABEL}_shared_data.pt"
data_settings = {
    "format_version": 5,
    "seed": SEED, "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED, "radius": R, "theta": theta, "sigma_squared": sigma_squared,
    "data_anchors": data_anchors, "paths_per_anchor": paths_per_anchor,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "training_collocation_count": training_collocation_points,
    "training_boundary_count": training_boundary_points,
    "evaluation_interior_count": evaluation_interior_points,
    "evaluation_boundary_count": evaluation_boundary_points,
    "lbfgs_interior_count": lbfgs_interior_points,
    "lbfgs_boundary_count": lbfgs_boundary_points,
    "n_test_functions": n_test_functions,
}

write_run_config("preparing")

if data_path.exists():
    if RUN_MODE == "new":
        raise FileExistsError(
            f"{data_path} already exists. Start a new run from Section 3; "
            "saved data will not be overwritten."
        )
    shared = torch.load(data_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(shared, data_settings, data_path)
    x_data, tau_data = shared["x"].to(DEVICE), shared["tau"].to(DEVICE)
    data_censored = shared["censored"].to(DEVICE)
    evaluation_interior = shared["evaluation_interior"].to(DEVICE)
    evaluation_boundary = shared["evaluation_boundary"].to(DEVICE)
    training_interior = shared["training_interior"].to(DEVICE)
    training_boundary = shared["training_boundary"].to(DEVICE)
    lbfgs_interior = shared["lbfgs_interior"].to(DEVICE)
    lbfgs_boundary = shared["lbfgs_boundary"].to(DEVICE)
    test_centers = shared["test_centers"].to(DEVICE)
    print("Loaded shared labels and evaluation points:", data_path)
else:
    if RUN_MODE != "new":
        raise FileNotFoundError(f"Saved shared data are missing from {output_root}.")
    random.seed(MC_SEED)
    np.random.seed(MC_SEED)
    torch.manual_seed(MC_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MC_SEED)
    x_data = sample_disk(data_anchors)
    tau_data, data_censored = monte_carlo_exit_time(x_data, paths_per_anchor, mc_dt, mc_max_time)
    evaluation_interior = sample_disk(evaluation_interior_points)
    evaluation_boundary = sample_boundary(evaluation_boundary_points)
    training_interior = sample_disk(training_collocation_points)
    training_boundary = sample_boundary(training_boundary_points)
    lbfgs_interior = sample_disk(lbfgs_interior_points)
    lbfgs_boundary = sample_boundary(lbfgs_boundary_points)
    shared = {
        "x": x_data.cpu(), "tau": tau_data.cpu(), "censored": data_censored.cpu(),
        "evaluation_interior": evaluation_interior.cpu(),
        "evaluation_boundary": evaluation_boundary.cpu(),
        "training_interior": training_interior.cpu(),
        "training_boundary": training_boundary.cpu(),
        "lbfgs_interior": lbfgs_interior.cpu(),
        "lbfgs_boundary": lbfgs_boundary.cpu(),
        "test_centers": test_centers.cpu(),
        "settings": data_settings,
    }
    temporary_data_path = data_path.with_name(data_path.name + ".tmp")
    torch.save(shared, temporary_data_path)
    temporary_data_path.replace(data_path)
    print("Saved shared labels and evaluation points:", data_path)

# Reset the training stream after label generation so new and resumed runs
# initialize their first model from the same recorded training seed.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

atomic_npz_save(
    output_root / "shared_data.npz",
    label_points=x_data.cpu().numpy(),
    label_values=tau_data.cpu().numpy(),
    censored_fraction=data_censored.cpu().numpy(),
    training_interior=training_interior.cpu().numpy(),
    training_boundary=training_boundary.cpu().numpy(),
    evaluation_interior=evaluation_interior.cpu().numpy(),
    evaluation_boundary=evaluation_boundary.cpu().numpy(),
    lbfgs_interior=lbfgs_interior.cpu().numpy(),
    lbfgs_boundary=lbfgs_boundary.cpu().numpy(),
    test_centers=test_centers.cpu().numpy(),
    training_seed=np.asarray(SEED, dtype=np.uint64),
    monte_carlo_seed=np.asarray(MC_SEED, dtype=np.uint64),
)

write_run_config("prepared")

print(f"Shared labels: {len(x_data)} anchors × {paths_per_anchor} paths")
print(f"Censored fraction: mean={float(data_censored.mean()):.2%}, max={float(data_censored.max()):.2%}")

### Train and save

The loop below makes one Adam update per method per epoch using every point in the saved training sets. It writes restart checkpoints and per-epoch CSV histories. The evaluation cell saves predictions, the analytic reference, error arrays, metrics, and plots as `.npz`, `.json`, and `.png` files.

In [ ]:
def atomic_torch_save(payload, path):
    temporary_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(path)


def atomic_csv_save(frame, path):
    temporary_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def atomic_npz_save(path, **arrays):
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary_path.replace(path)


def atomic_figure_save(figure, path, **kwargs):
    temporary_path = path.with_name(path.stem + ".tmp" + path.suffix)
    figure.savefig(temporary_path, **kwargs)
    temporary_path.replace(path)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    cuda_state = state.get("cuda")
    if cuda_state is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([item.cpu() for item in cuda_state])


def detached_row(components):
    return {name: float(value.detach()) for name, value in components.items()}


def train_method(method):
    run_dir = output_root / f"{RUN_LABEL}_{method}"
    run_dir.mkdir(parents=True, exist_ok=True)
    model = OUModel(method).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(adam_beta_1, adam_beta_2),
        eps=adam_epsilon,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer,
        gamma=final_learning_rate_factor ** (1.0 / adam_epochs),
    )
    history, start_epoch, best_score = [], 0, math.inf
    last_path, best_path = run_dir / "last.pt", run_dir / "best_adam.pt"
    final_path = run_dir / "final.pt"
    lbfgs_path = run_dir / "lbfgs_history.csv"
    lbfgs_history = []
    checkpoint_settings = {**experiment_settings, "method": method}
    final_settings = {
        **checkpoint_settings, "lbfgs": RUN_LBFGS,
        "lbfgs_iterations": lbfgs_iterations,
    }

    if RESUME and final_path.exists():
        state = torch.load(final_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, final_settings, final_path)
        if "rng_state" not in state:
            raise ValueError(
                f"{final_path} predates saved RNG state and cannot be resumed faithfully. "
                "Use RUN_MODE='evaluate' for its model, or start a new run."
            )
        model.load_state_dict(state["model"])
        restore_rng_state(state["rng_state"])
        return model, pd.DataFrame(state["history"])

    if not RESUME and any(path.exists() for path in (last_path, best_path, final_path)):
        raise FileExistsError(
            f"Training files already exist in {run_dir}. Choose RUN_MODE='resume' "
            "with this run folder, or start a new run."
        )

    if RESUME and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, checkpoint_settings, last_path)
        if "rng_state" not in state:
            raise ValueError(
                f"{last_path} predates saved RNG state and cannot be resumed faithfully. "
                "Use RUN_MODE='evaluate' for its final model, or start a new run."
            )
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_score = state["history"], state["epoch"], state["best_score"]
        restore_rng_state(state["rng_state"])
        print(f"{method}: resuming after epoch {start_epoch}")
    elif RESUME and best_path.exists():
        raise FileNotFoundError(
            f"{run_dir} has a best checkpoint but no restart checkpoint. "
            "Start a new run or evaluate a completed final.pt."
        )

    started = time.time()
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        model.train()
        optimizer.zero_grad()
        interior = training_interior
        boundary = training_boundary
        components = component_losses(model, method, interior, boundary, x_data, tau_data)
        objective = adaptive_objective(model, components)
        objective.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        validation = component_losses(model, method, evaluation_interior, evaluation_boundary, x_data, tau_data)
        score = float(adaptive_objective(model, validation).detach())
        row = {
            "epoch": epoch,
            "objective": float(objective.detach()),
            "checkpoint_objective": score,
            **detached_row(components),
        }
        history.append(row)
        if score < best_score:
            best_score = score
            atomic_torch_save(
                {"model": model.state_dict(), "epoch": epoch, "score": score, "settings": checkpoint_settings},
                best_path,
            )

        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            pieces = " | ".join(f"{key} {value:.3e}" for key, value in detached_row(components).items())
            print(
                f"{method:11s} epoch {epoch:5d}/{adam_epochs} | {pieces} | "
                f"{time.time() - started:.1f}s",
                flush=True,
            )
        if epoch % checkpoint_every == 0 or epoch == 1 or epoch == adam_epochs:
            atomic_torch_save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "history": history,
                "epoch": epoch, "best_score": best_score,
                "settings": checkpoint_settings, "rng_state": capture_rng_state(),
            }, last_path)
            atomic_csv_save(pd.DataFrame(history), run_dir / "training_metrics.csv")
            write_run_config("training")

    best_state = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_state["model"])
    if RUN_LBFGS:
        fixed_interior = lbfgs_interior
        fixed_boundary = lbfgs_boundary
        tolerance = lbfgs_tolerance
        lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=lbfgs_lr,
            max_iter=lbfgs_iterations,
            history_size=lbfgs_history_size,
            tolerance_grad=tolerance,
            tolerance_change=tolerance,
            line_search_fn="strong_wolfe",
        )
        calls = 0

        def closure():
            nonlocal calls
            calls += 1
            lbfgs.zero_grad()
            parts = component_losses(model, method, fixed_interior, fixed_boundary, x_data, tau_data)
            loss = adaptive_objective(model, parts)
            loss.backward()
            lbfgs_history.append({"call": calls, "objective": float(loss.detach()), **detached_row(parts)})
            if calls % 100 == 0:
                atomic_csv_save(pd.DataFrame(lbfgs_history), lbfgs_path)
                write_run_config("training")
                print(method, "L-BFGS call", calls, detached_row(parts), flush=True)
            return loss

        lbfgs.step(closure)
        atomic_csv_save(pd.DataFrame(lbfgs_history), lbfgs_path)
    atomic_torch_save({
        "model": model.state_dict(), "history": history, "method": method,
        "run_label": RUN_LABEL, "settings": final_settings,
        "rng_state": capture_rng_state(),
        "lbfgs_history": lbfgs_history,
    }, final_path)
    return model, pd.DataFrame(history)

In [ ]:
trained = {}
histories = {}
if RUN_MODE in {"new", "resume"}:
    write_run_config("training")
    for method in METHODS:
        trained[method], histories[method] = train_method(method)
    print("Training complete:", ", ".join(trained))
    write_run_config("trained")
elif RUN_MODE == "evaluate":
    write_run_config("evaluating")
    for method in METHODS:
        run_dir = output_root / f"{RUN_LABEL}_{method}"
        final_path = run_dir / "final.pt"
        state = torch.load(final_path, map_location=DEVICE, weights_only=False)
        expected = {
            **experiment_settings, "method": method, "lbfgs": RUN_LBFGS,
            "lbfgs_iterations": lbfgs_iterations,
        }
        require_matching_settings(state, expected, final_path)
        model = OUModel(method).to(DEVICE)
        model.load_state_dict(state["model"])
        trained[method] = model
        histories[method] = pd.DataFrame(state["history"])
    print("Loaded final models:", ", ".join(trained))

if trained:
    for method, history in histories.items():
        run_dir = output_root / f"{RUN_LABEL}_{method}"
        run_dir.mkdir(parents=True, exist_ok=True)
        metrics_path = run_dir / "training_metrics.csv"
        temporary_metrics_path = metrics_path.with_name(metrics_path.name + ".tmp")
        history.to_csv(temporary_metrics_path, index=False)
        temporary_metrics_path.replace(metrics_path)
        if RUN_LBFGS:
            final_path = run_dir / "final.pt"
            final_state = torch.load(final_path, map_location=DEVICE, weights_only=False)
            lbfgs_frame = pd.DataFrame(final_state.get("lbfgs_history", []))
            if not lbfgs_frame.empty:
                atomic_csv_save(lbfgs_frame, run_dir / "lbfgs_history.csv")
    print("Per-epoch training metrics saved under:", output_root)

## 7. Evaluate and report

The notebook compares each model with the radial reference solution and reports relative $L^1$, $L^2$, $H^1$, and $H^2$ errors on the configured grid.

In [ ]:
experiment_settings = {
    "run_label": RUN_LABEL, "seed": SEED, "mc_seed": MC_SEED,
    "radius": R, "theta": theta, "sigma_squared": sigma_squared,
    "training_collocation_points": training_collocation_points,
    "training_boundary_points": training_boundary_points,
    "evaluation_interior_points": evaluation_interior_points,
    "evaluation_boundary_points": evaluation_boundary_points,
    "lbfgs_interior_points": lbfgs_interior_points, "lbfgs_boundary_points": lbfgs_boundary_points,
    "data_anchors": data_anchors, "paths_per_anchor": paths_per_anchor,
    "mc_dt": mc_dt, "mc_max_time": mc_max_time,
    "adam_epochs": adam_epochs,
    "learning_rate": learning_rate, "final_learning_rate_factor": final_learning_rate_factor,
    "adam_beta_1": adam_beta_1, "adam_beta_2": adam_beta_2,
    "adam_epsilon": adam_epsilon, "weight_decay": weight_decay,
    "methods": list(METHODS), "run_lbfgs": RUN_LBFGS,
    "lbfgs_iterations": lbfgs_iterations, "lbfgs_lr": lbfgs_lr,
    "n_test_functions": n_test_functions, "test_length": test_length,
    "reference_grid_n": reference_grid_n, "device": str(DEVICE), "dtype": DTYPE_NAME,
    "trajectory_start": trajectory_start, "trajectory_dt": trajectory_dt,
    "trajectory_max_time": trajectory_max_time, "trajectory_seed_offset": trajectory_seed_offset,
    "adaptive_log_scale_init": adaptive_log_scale_init, "nitsche_bc_log_scale_init": nitsche_bc_log_scale_init,
}
experiment_settings = json.loads(json.dumps(experiment_settings))

def require_matching_settings(record, expected, saved_path):
    saved = record.get("settings", record)
    if saved != expected:
        raise ValueError(
            f"{saved_path} was created with different settings. "
            "Restore those settings or start a new run."
        )

manifest_path = output_root / "run_manifest.json"
if RUN_MODE == "new":
    write_run_config_record = {
        "schema_version": 2,
        "run_id": output_root.name,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "seed": SEED,
        "mc_seed": MC_SEED,
        "monte_carlo_seed": MC_SEED,
        "settings": experiment_settings,
        "status": "created",
        "artifacts": [],
    }
    with manifest_path.open("x", encoding="utf-8") as stream:
        json.dump(write_run_config_record, stream, indent=2)
else:
    write_run_config_record = json.loads(manifest_path.read_text(encoding="utf-8"))
    require_matching_settings(write_run_config_record, experiment_settings, manifest_path)

def write_run_config(status):
    write_run_config_record["status"] = status
    write_run_config_record["updated_utc"] = datetime.now(timezone.utc).isoformat()
    write_run_config_record["artifacts"] = sorted(
        str(path.relative_to(output_root))
        for path in output_root.rglob("*")
        if path.is_file() and path != manifest_path and ".tmp" not in path.name
    )
    temporary_path = manifest_path.with_name(manifest_path.name + ".tmp")
    temporary_path.write_text(json.dumps(write_run_config_record, indent=2), encoding="utf-8")
    temporary_path.replace(manifest_path)

def atomic_npz_save(path, **arrays):
    temporary_path = path.with_name(path.name + ".tmp")
    with temporary_path.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary_path.replace(path)

def atomic_figure_save(figure, path, **kwargs):
    temporary_path = path.with_name(path.stem + ".tmp" + path.suffix)
    figure.savefig(temporary_path, **kwargs)
    temporary_path.replace(path)

def require_artifact_files(paths):
    missing = [path for path in paths if not path.is_file() or path.stat().st_size == 0]
    if missing:
        raise RuntimeError("Missing or empty run outputs: " + ", ".join(map(str, missing)))

write_run_config("preparing")
print(f"Run folder: {output_root} | seed: {SEED} | Monte Carlo seed: {MC_SEED}")


### Optional trajectory picture

This one-path plot is a visual check of the OU simulation. It does not enter training or the Monte Carlo labels.

In [ ]:
def evaluate_model(model):
    model.eval()
    with torch.no_grad():
        prediction = model(torch.as_tensor(grid_points, device=DEVICE)).squeeze(1).cpu().numpy()
    predicted_grid = np.zeros_like(gx)
    predicted_grid[disk_mask] = prediction
    truth = reference_grid[disk_mask]
    spacing = 2 * R / (grid_n - 1)
    error_grid = predicted_grid - reference_grid
    error_y, error_x = np.gradient(error_grid, spacing, spacing)
    error_xx = np.gradient(error_x, spacing, axis=1)
    error_yy = np.gradient(error_y, spacing, axis=0)
    error_xy = np.gradient(error_y, spacing, axis=1)
    truth_y, truth_x = np.gradient(reference_grid, spacing, spacing)
    truth_xx = np.gradient(truth_x, spacing, axis=1)
    truth_yy = np.gradient(truth_y, spacing, axis=0)
    truth_xy = np.gradient(truth_y, spacing, axis=1)
    rel_l1 = np.mean(np.abs(prediction - truth)) / np.mean(np.abs(truth))
    rel_l2 = np.sqrt(np.mean((prediction - truth)**2) / np.mean(truth**2))
    h1_error = np.mean(
        error_grid[disk_mask]**2
        + error_x[disk_mask]**2
        + error_y[disk_mask]**2
    )
    h1_reference = np.mean(
        reference_grid[disk_mask]**2
        + truth_x[disk_mask]**2
        + truth_y[disk_mask]**2
    )
    rel_h1 = np.sqrt(h1_error / h1_reference)
    h2_error = np.mean(
        error_grid[disk_mask]**2
        + error_x[disk_mask]**2
        + error_y[disk_mask]**2
        + error_xx[disk_mask]**2
        + error_yy[disk_mask]**2
        + 2 * error_xy[disk_mask]**2
    )
    h2_reference = np.mean(
        reference_grid[disk_mask]**2
        + truth_x[disk_mask]**2
        + truth_y[disk_mask]**2
        + truth_xx[disk_mask]**2
        + truth_yy[disk_mask]**2
        + 2 * truth_xy[disk_mask]**2
    )
    rel_h2 = np.sqrt(h2_error / h2_reference)
    metrics = {
        "L1 relative": float(rel_l1),
        "L2 relative": float(rel_l2),
        "H1 relative": float(rel_h1),
        "H2 relative": float(rel_h2),
    }
    return predicted_grid, metrics


if trained:
    write_run_config("evaluating")
    figures = {"reference": reference_grid}
    metrics = {}
    error_fields = {}
    for name, model in trained.items():
        figures[name], metrics[name] = evaluate_model(model)
        error_fields[name] = figures[name] - reference_grid

    display(pd.DataFrame(metrics).T)

    result_arrays = {
        "gx": gx,
        "gy": gy,
        "grid_points": grid_points,
        "grid_spacing": np.asarray(2 * R / (grid_n - 1)),
        "grid_size": np.asarray(grid_n),
        "disk_mask": disk_mask,
        "reference_grid": reference_grid,
        "monte_carlo_anchors": x_data.detach().cpu().numpy(),
        "monte_carlo_exit_times": tau_data.detach().cpu().numpy(),
        "monte_carlo_censored_fraction": data_censored.detach().cpu().numpy(),
        "evaluation_interior": evaluation_interior.detach().cpu().numpy(),
        "evaluation_boundary": evaluation_boundary.detach().cpu().numpy(),
        "training_interior": training_interior.detach().cpu().numpy(),
        "training_boundary": training_boundary.detach().cpu().numpy(),
        "lbfgs_interior": lbfgs_interior.detach().cpu().numpy(),
        "lbfgs_boundary": lbfgs_boundary.detach().cpu().numpy(),
        "nitsche_test_centers": test_centers.detach().cpu().numpy(),
        "run_seed": np.asarray(SEED, dtype=np.uint64),
        "monte_carlo_seed": np.asarray(MC_SEED, dtype=np.uint64),
    }
    for name, prediction in figures.items():
        if name != "reference":
            result_arrays[f"prediction_{name}"] = prediction
            result_arrays[f"signed_error_{name}"] = error_fields[name]
    for name, history in histories.items():
        for column in history.select_dtypes(include=[np.number]).columns:
            result_arrays[f"training_{name}_{column}"] = history[column].to_numpy()

    for method in trained:
        lbfgs_path = output_root / f"{RUN_LABEL}_{method}" / "lbfgs_history.csv"
        if lbfgs_path.is_file():
            lbfgs_frame = pd.read_csv(lbfgs_path)
            for column in lbfgs_frame.select_dtypes(include=[np.number]).columns:
                result_arrays[f"lbfgs_{method}_{column}"] = lbfgs_frame[column].to_numpy()

    results_path = output_root / "numerical_results.npz"
    temporary_results_path = results_path.with_name(results_path.name + ".tmp")
    with temporary_results_path.open("wb") as handle:
        np.savez_compressed(handle, **result_arrays)
    temporary_results_path.replace(results_path)

    error_limit = max(
        float(np.max(np.abs(values[disk_mask])))
        for values in error_fields.values()
    )
    error_limit = max(error_limit, np.finfo(float).eps)
    solution_values = [figures[name][disk_mask] for name in figures]
    solution_min = min(float(values.min()) for values in solution_values)
    solution_max = max(float(values.max()) for values in solution_values)

    metrics_record = {
        "run_id": output_root.name,
        "seed": SEED,
        "settings": experiment_settings,
        "methods": list(trained),
        "metrics": metrics,
        "solution_color_limits": [solution_min, solution_max],
        "absolute_error_color_limits": [0.0, error_limit],
        "arrays_file": results_path.name,
        "training_metrics_files": {
            name: f"{RUN_LABEL}_{name}/training_metrics.csv" for name in histories
        },
    }
    metrics_path = output_root / "numerical_metrics.json"
    temporary_metrics_path = metrics_path.with_name(metrics_path.name + ".tmp")
    temporary_metrics_path.write_text(json.dumps(metrics_record, indent=2) + "\n", encoding="utf-8")
    temporary_metrics_path.replace(metrics_path)

    solution_levels = np.linspace(solution_min, solution_max, 51)
    solution_figure, solution_axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (name, values) in zip(solution_axes.flat, figures.items()):
        plot = axis.contourf(
            gx, gy, np.ma.masked_where(~disk_mask, values),
            levels=solution_levels, vmin=solution_min, vmax=solution_max, extend="both",
        )
        axis.set_title(name)
        axis.set_aspect("equal")
        solution_figure.colorbar(plot, ax=axis)
    for axis in solution_axes.flat[len(figures):]:
        axis.axis("off")
    solution_figure.suptitle("OU mean escape time: solution fields")
    atomic_figure_save(solution_figure, output_root / "solution_fields.png", dpi=plot_dpi)
    plt.show()

    error_levels = np.linspace(0.0, error_limit, 51)
    error_figure, error_axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for axis, (name, values) in zip(error_axes.flat, error_fields.items()):
        plot = axis.contourf(
            gx, gy, np.ma.masked_where(~disk_mask, np.abs(values)),
            levels=error_levels, vmin=0.0, vmax=error_limit, cmap="magma", extend="max",
        )
        axis.set_title(f"{name}: absolute error")
        axis.set_aspect("equal")
        error_figure.colorbar(plot, ax=axis)
    for axis in error_axes.flat[len(error_fields):]:
        axis.axis("off")
    error_figure.suptitle(f"OU mean escape time: shared absolute-error scale [0, {error_limit:.3g}]")
    atomic_figure_save(error_figure, output_root / "absolute_error_fields.png", dpi=plot_dpi)
    plt.show()

    write_run_config("evaluating")
    required_paths = [
        data_path, manifest_path, results_path, metrics_path,
        output_root / "solution_fields.png",
        output_root / "absolute_error_fields.png",
    ]
    for method in trained:
        method_dir = output_root / f"{RUN_LABEL}_{method}"
        required_paths.extend([
            method_dir / "best_adam.pt",
            method_dir / "last.pt",
            method_dir / "final.pt",
            method_dir / "training_metrics.csv",
        ])
        if RUN_LBFGS:
            required_paths.append(method_dir / "lbfgs_history.csv")
    missing_paths = [path for path in required_paths if not path.is_file() or path.stat().st_size == 0]
    if missing_paths:
        missing_list = ", ".join(map(str, missing_paths))
        raise RuntimeError(
            "Run outputs are incomplete; missing or empty files: " + missing_list
        )
    required_keys = {
        "gx", "gy", "grid_points", "disk_mask", "reference_grid",
        "monte_carlo_anchors", "monte_carlo_exit_times", "monte_carlo_seed",
        "monte_carlo_censored_fraction", "evaluation_interior",
        "evaluation_boundary", "training_interior", "training_boundary",
        "lbfgs_interior", "lbfgs_boundary", "run_seed", "monte_carlo_seed",
    }
    for method in trained:
        required_keys.update({f"prediction_{method}", f"signed_error_{method}", f"training_{method}_epoch"})
    with np.load(results_path, allow_pickle=False) as saved_arrays:
        missing_keys = required_keys.difference(saved_arrays.files)
    if missing_keys:
        raise RuntimeError("Result archive is incomplete; missing arrays: " + ", ".join(sorted(missing_keys)))
    write_run_config("complete")
    print("Validated run outputs and saved artifact inventory under:", output_root)

### Previously reported comparison

The table below is a previously reported comparison, included for context. New runs save the reference grid, predictions, signed errors, Monte Carlo labels, evaluation points, and per-epoch metrics, so figures and additional metrics can be recomputed without retraining.

## 8. Reuse and adapt

The problem parameters are in the settings cells near the start. The sampler, stochastic simulator, neural network, differential residual, optimizer, and reporting code are each defined in their own cell. Edit the relevant code directly when changing the problem or training method.

## References

- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- H. Kersting, A. Orvieto, F. Proske, and A. Lucchi, [“Mean first exit times of Ornstein–Uhlenbeck processes in high-dimensional spaces”](https://arxiv.org/abs/2208.04029), *J. Phys. A* 56 (2023), 215003.
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- Y. Liao and P. Ming, [“Deep Nitsche Method”](https://doi.org/10.4208/cicp.OA-2020-0219), *Commun. Comput. Phys.* 29 (2021).
- A. Kendall, Y. Gal, and R. Cipolla, [“Multi-task learning using uncertainty to weigh losses”](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html), CVPR 2018.
